# fabric-medallion-dbt

One step of `run_pipeline`, which is what you run or schedule:

```
land  ->  dwh   ->  parity
      ->  spark ->
```

`land` downloads the AEMO files once, `dwh` and `spark` build the same gold layer on Fabric
Warehouse and Fabric Spark, and `parity` fails the run if the two disagree.

Settings are in the `deploy_config` Variable Library: `engines` (`all`, `dwh` or `spark`),
the download and process limits, and `repo_ref`, the branch or tag of
[fabric-medallion-dbt](https://github.com/djouallah/fabric-medallion-dbt) this notebook runs.

In [ ]:
# Set by run_pipeline. Run by hand, this lands the files; change `step` to build an engine.
step = "land"       # land | dwh | spark | parity
run_id = "manual"

In [ ]:
# Nothing dbt-shaped lives here. This notebook fetches the repo and runs fabric_run.py, the
# same commands the project's CI ran; change the scripts, not the notebook.
import io
import os
import shutil
import subprocess
import sys
import urllib.request
import zipfile

import notebookutils

REPO_URL = "https://github.com/djouallah/fabric-medallion-dbt"

vl = notebookutils.variableLibrary.getLibrary("deploy_config")
for k in ("engines", "process_limit", "download_limit", "daily_download_limit"):
    os.environ[k] = getattr(vl, k)
os.environ["DBT_SCHEMA"] = vl.dbt_schema
os.environ["SPARK_RESOURCE_PROFILE"] = vl.spark_resource_profile
os.environ["FABRIC_WORKSPACE_ID"] = notebookutils.runtime.context["currentWorkspaceId"]
# The workspace folder the install created; provision.py keeps the lakehouses and the
# warehouse in it, next to this notebook.
os.environ["FOLDER"] = "fabric-medallion-dbt"

# The ~135 GiB work disk, not the 19 GiB /tmp overlay.
work = "/home/trusted-service-user/work" if os.path.isdir("/home/trusted-service-user/work") else "/tmp"
with urllib.request.urlopen(f"{REPO_URL}/archive/{vl.repo_ref}.zip") as r:
    archive = zipfile.ZipFile(io.BytesIO(r.read()))
# GitHub names the one top-level folder <repo>-<ref>.
repo = os.path.join(work, archive.namelist()[0].split("/")[0])
shutil.rmtree(repo, ignore_errors=True)
archive.extractall(work)
print(f"{REPO_URL} @ {vl.repo_ref} -> {repo}")

In [ ]:
# Streamed line by line, so the dbt log is in this cell's output while it runs.
p = subprocess.Popen([sys.executable, "-u", "fabric_run.py", step, run_id], cwd=repo,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
if p.wait():
    # Raise so a scheduled run goes red instead of silently reporting success.
    raise RuntimeError(f"{step} failed (exit {p.returncode}); see the log above")